# Redo of Beta / Free variable substitution I did awhile ago.

## Grading Suite

In [ ]:
"""
Grading suite for: Lambda Calculus Beta Reduction (beta_reduce)

This file provides:
  1. A minimal parser -> AST, so the grader can compare results
     structurally rather than by raw string match.
  2. An alpha-equivalence checker (two expressions are "equal" if they
     differ only by consistent renaming of bound variables).
  3. A structured test case table covering normal cases, edge cases,
     and adversarial/performance cases.
  4. A test runner that reports pass/fail, timing, and step-count info.

NOTE: `beta_reduce` itself is NOT implemented here. Import your own
implementation and point SOLUTION_FN at it, or paste your solution
into solution.py and this file will pick it up automatically.
"""

import re
import time
import traceback
from dataclasses import dataclass
from typing import Union, Optional


# ---------------------------------------------------------------------------
# 1. AST + parser (grading infrastructure only -- not the interpreter itself)
# ---------------------------------------------------------------------------

@dataclass(frozen=True)
class Var:
    name: str

@dataclass(frozen=True)
class Abs:
    param: str
    body: "Expr"

@dataclass(frozen=True)
class App:
    fn: "Expr"
    arg: "Expr"

Expr = Union[Var, Abs, App]

VAR_RE = re.compile(r"[a-z][a-z0-9']*")
TOKEN_RE = re.compile(r"\s*(λ|\\|lambda|\(|\)|\.|[a-z][a-z0-9']*)\s*")


class ParseError(Exception):
    pass


def tokenize(s: str):
    pos = 0
    tokens = []
    while pos < len(s):
        m = TOKEN_RE.match(s, pos)
        if not m or m.end() == pos:
            if s[pos:].strip() == "":
                break
            raise ParseError(f"Unexpected character at position {pos}: {s[pos:pos+10]!r}")
        tok = m.group(1)
        if tok:
            tokens.append(tok)
        pos = m.end()
    return tokens


class Parser:
    """
    Grammar (matches the problem statement):
        expr := VAR
              | '(' 'λ' VAR '.' expr ')'
              | '(' expr expr ')'
    """
    def __init__(self, tokens):
        self.tokens = tokens
        self.i = 0

    def peek(self):
        return self.tokens[self.i] if self.i < len(self.tokens) else None

    def advance(self):
        tok = self.peek()
        self.i += 1
        return tok

    def expect(self, tok):
        got = self.advance()
        if got != tok:
            raise ParseError(f"Expected {tok!r}, got {got!r}")

    def parse_expr(self) -> Expr:
        tok = self.peek()
        if tok is None:
            raise ParseError("Unexpected end of input")
        if VAR_RE.fullmatch(tok) and tok not in ("λ", "lambda"):
            self.advance()
            return Var(tok)
        if tok == "(":
            self.advance()
            if self.peek() in ("λ", "\\", "lambda"):
                self.advance()
                param = self.advance()
                if not VAR_RE.fullmatch(param or ""):
                    raise ParseError(f"Expected variable name, got {param!r}")
                self.expect(".")
                body = self.parse_expr()
                self.expect(")")
                return Abs(param, body)
            else:
                fn = self.parse_expr()
                arg = self.parse_expr()
                self.expect(")")
                return App(fn, arg)
        raise ParseError(f"Unexpected token {tok!r}")

    def parse(self) -> Expr:
        expr = self.parse_expr()
        if self.i != len(self.tokens):
            raise ParseError(f"Trailing tokens: {self.tokens[self.i:]}")
        return expr


def parse(s: str) -> Expr:
    return Parser(tokenize(s)).parse()


# ---------------------------------------------------------------------------
# 2. Alpha-equivalence checker
# ---------------------------------------------------------------------------

def alpha_equivalent(a: Expr, b: Expr, env_a=None, env_b=None, depth=0) -> bool:
    """
    Structural equality up to consistent renaming of bound variables.
    env_a/env_b map bound-variable-name -> de-Bruijn-style depth at
    which it was introduced, so shadowing and reuse are handled correctly.
    """
    env_a = env_a or {}
    env_b = env_b or {}

    if isinstance(a, Var) and isinstance(b, Var):
        bound_a = env_a.get(a.name)
        bound_b = env_b.get(b.name)
        if bound_a is None and bound_b is None:
            # both free -- must be the same free variable
            return a.name == b.name
        # both must be bound at the same "depth" (same binder position)
        return bound_a == bound_b

    if isinstance(a, Abs) and isinstance(b, Abs):
        new_env_a = {**env_a, a.param: depth}
        new_env_b = {**env_b, b.param: depth}
        return alpha_equivalent(a.body, b.body, new_env_a, new_env_b, depth + 1)

    if isinstance(a, App) and isinstance(b, App):
        return (alpha_equivalent(a.fn, b.fn, env_a, env_b, depth) and
                alpha_equivalent(a.arg, b.arg, env_a, env_b, depth))

    return False


def expr_equal(expected_str: str, actual_str: str) -> bool:
    """Grading equality: parse both sides and compare up to alpha-renaming."""
    if actual_str.strip() == "DIVERGES":
        return expected_str.strip() == "DIVERGES"
    try:
        return alpha_equivalent(parse(expected_str), parse(actual_str))
    except ParseError as e:
        raise AssertionError(f"Output is not parseable as a valid expression: {e}")


# ---------------------------------------------------------------------------
# 3. Test cases
# ---------------------------------------------------------------------------

@dataclass
class TestCase:
    name: str
    expression: str
    expected: str
    max_steps: int = 1000
    timeout_s: float = 2.0
    category: str = "general"


TEST_CASES = [
    # --- Basic reduction ---
    TestCase("identity_application", "((λx. x) y)", "y", category="basic"),
    TestCase("no_redex_free_var", "x", "x", category="basic"),
    TestCase("no_redex_open_app", "(x y)", "(x y)", category="basic"),
    TestCase("nested_abstraction_subst", "((λx. (λy. x)) a)", "(λy. a)", category="basic"),

    # --- Capture avoidance ---
    TestCase("capture_avoidance_simple", "((λx. (λy. x)) y)", "(λy'. y)", category="capture"),
    TestCase(
        "capture_avoidance_nested",
        "((λx. (λy. (λz. x))) y)",
        "(λy'. (λz. y))",
        category="capture",
    ),
    TestCase(
        "shadowing_not_touched",
        "((λx. (λx. x)) y)",
        "(λx. x)",
        category="capture",
        # inner x shadows outer -- substitution must NOT reach inner x
    ),
    TestCase(
        "fresh_name_already_taken",
        # renaming y -> y' must itself avoid collision if y' is also in play
        "((λx. (λy. (λy'. x))) y)",
        "(λy''. (λy'. y))",
        category="capture",
    ),

    # --- Vacuous binding / multiple occurrences ---
    TestCase("vacuous_binding", "((λx. y) z)", "y", category="substitution"),
    TestCase("multiple_occurrences", "((λx. (x x)) y)", "(y y)", category="substitution"),
    TestCase(
        "argument_is_abstraction",
        "((λf. (f a)) (λx. x))",
        "a",
        category="substitution",
    ),

    # --- Reduction order matters ---
    TestCase(
        "normal_order_terminates",
        # applicative order would diverge trying to evaluate the argument first;
        # normal order discards it via vacuous binding and terminates
        "((λx. y) ((λz. (z z)) (λz. (z z))))",
        "y",
        category="order",
    ),
    TestCase(
        "self_application_terminates",
        "((λf. (f f)) (λx. x))",
        "(λx. x)",
        category="order",
    ),

    # --- Divergence ---
    TestCase("omega_combinator", "((λx. (x x)) (λx. (x x)))", "DIVERGES",
              max_steps=50, category="divergence"),
    TestCase("diverges_at_boundary_low", "((λx. (x x)) (λx. (x x)))", "DIVERGES",
              max_steps=1, category="divergence"),

    # --- Malformed input (grader expects an exception, not a crash-y result) ---
    TestCase("malformed_unbalanced_paren", "((λx. x)", "__RAISES__", category="malformed"),
    TestCase("malformed_empty", "", "__RAISES__", category="malformed"),
    TestCase("malformed_bad_var", "((λ1. x) y)", "__RAISES__", category="malformed"),

    # --- Performance / adversarial ---
    TestCase(
        "church_numeral_addition",
        # PLUS 2 3 in Church encoding, expect Church numeral 5 in normal form
        "((((λm. (λn. (λf. (λx. ((m f) ((n f) x)))))) "
        "(λf. (λx. (f (f x))))) "
        "(λf. (λx. (f (f (f x)))))) )",  # left intentionally illustrative; fill in real encoding
        "(λf. (λx. (f (f (f (f (f x)))))))",
        max_steps=200,
        timeout_s=1.0,
        category="performance",
    ),
    TestCase(
        "deep_left_nested_application",
        "(" * 20 + "a " + "b) " * 20,  # stress parser recursion depth; adjust to valid grammar
        "__STRUCTURAL_CHECK_ONLY__",
        category="performance",
    ),
]


# ---------------------------------------------------------------------------
# 4. Test runner
# ---------------------------------------------------------------------------

def run_suite(solution_fn, cases=TEST_CASES, verbose=True):
    results = []
    for tc in cases:
        start = time.perf_counter()
        try:
            if tc.expected == "__RAISES__":
                try:
                    solution_fn(tc.expression, tc.max_steps)
                    ok, detail = False, "expected an exception, got a normal return"
                except Exception:
                    ok, detail = True, "raised as expected"
            elif tc.expected == "__STRUCTURAL_CHECK_ONLY__":
                out = solution_fn(tc.expression, tc.max_steps)
                parse(out)  # just confirm it's a well-formed expression
                ok, detail = True, f"parsed OK: {out[:60]}..."
            else:
                out = solution_fn(tc.expression, tc.max_steps)
                ok = expr_equal(tc.expected, out)
                detail = f"got {out!r}"
        except Exception as e:
            ok, detail = False, f"raised unexpectedly: {e}\n{traceback.format_exc()}"
        elapsed = time.perf_counter() - start
        timed_out = elapsed > tc.timeout_s
        results.append((tc, ok and not timed_out, detail, elapsed))
        if verbose:
            status = "PASS" if ok and not timed_out else "FAIL"
            extra = " (TIMEOUT)" if timed_out else ""
            print(f"[{status}]{extra} {tc.category:12s} {tc.name:30s} ({elapsed*1000:.1f}ms) {detail}")

    passed = sum(1 for _, ok, _, _ in results if ok)
    print(f"\n{passed}/{len(results)} passed")
    return results


if __name__ == "__main__":
    def beta_reduce_stub(expression: str, max_steps: int = 1000) -> str:
        raise NotImplementedError("Plug in your beta_reduce implementation here")

    run_suite(beta_reduce_stub)

In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from typing import TypeAlias

# Core task overview:
# 1. Parse the input string into a small lambda-calculus AST.
# 2. Compute free variables so substitution can detect capture risks.
# 3. Generate fresh variable names when alpha-renaming is required.
# 4. Implement capture-avoiding substitution.
# 5. Reduce one leftmost-outermost beta redex at a time.
# 6. Repeatedly step until normal form or max_steps is reached.
# 7. Expose `solution(expression, max_steps)` as the notebook entrypoint.


@dataclass(frozen=True)
class Var:
    name: str


@dataclass(frozen=True)
class Abs:
    param: str
    body: Expr


@dataclass(frozen=True)
class App:
    fn: Expr
    arg: Expr


Expr: TypeAlias = Var | Abs | App


def parse_expr(expression: str) -> Expr:
    ...


def free_vars(expr: Expr) -> set[str]:
    ...


def fresh_name(base: str, forbidden: set[str]) -> str:
    ...


def substitute(expr: Expr, var: str, replacement: Expr) -> Expr:
    ...


def beta_reduce_once(expr: Expr) -> tuple[Expr, bool]:
    ...


def render_expr(expr: Expr) -> str:
    ...


def solution(expression: str, max_steps: int = 1000) -> str:
    ...
